# Task 1 Quick Reorder Recommender

This notebook orchestrates the Task 1 pipeline. Core logic lives in `src/recommender/`.

The dataset is a fake/synthetic DESD seed export, not real customer behaviour.

In [2]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import TASK1_DATA_DIR, TASK1_OUTPUT_DIR
from src.recommender.data_loader import build_order_lines, dataset_summary, load_task1_dataset
from src.recommender.pipeline import run_task1_evaluation

print('Data folder:', TASK1_DATA_DIR)
print('Output folder:', TASK1_OUTPUT_DIR)

Data folder: D:\UWE\3rd Year\2nd semester\advanced_ai\ai_system\data\task1\desd_seed_export
Output folder: D:\UWE\3rd Year\2nd semester\advanced_ai\ai_system\outputs\task1_recommender


In [3]:
dataset = load_task1_dataset(TASK1_DATA_DIR)
order_lines = build_order_lines(dataset)
summary = dataset_summary(dataset, order_lines)
summary

{'dataset_type': 'synthetic_desd_seed_export',
 'customers': 80,
 'producers': 12,
 'products': 60,
 'orders': 483,
 'order_lines': 1794,
 'date_range': {'start': '2026-01-10', 'end': '2026-05-01'},
 'source_folder': 'D:\\UWE\\3rd Year\\2nd semester\\advanced_ai\\ai_system\\data\\task1\\desd_seed_export',
 'limitations': ['synthetic_seed_data_not_real_customer_behaviour',
  'metrics_are_proof_of_concept_not_production_accuracy']}

In [4]:
written = run_task1_evaluation(TASK1_DATA_DIR, TASK1_OUTPUT_DIR, top_k=3)
written

{'recommender_metrics.csv': 'D:\\UWE\\3rd Year\\2nd semester\\advanced_ai\\ai_system\\outputs\\task1_recommender\\recommender_metrics.csv',
 'recommendation_examples.csv': 'D:\\UWE\\3rd Year\\2nd semester\\advanced_ai\\ai_system\\outputs\\task1_recommender\\recommendation_examples.csv',
 'producer_diversity.csv': 'D:\\UWE\\3rd Year\\2nd semester\\advanced_ai\\ai_system\\outputs\\task1_recommender\\producer_diversity.csv',
 'product_coverage.csv': 'D:\\UWE\\3rd Year\\2nd semester\\advanced_ai\\ai_system\\outputs\\task1_recommender\\product_coverage.csv',
 'recommendation_share_by_producer.csv': 'D:\\UWE\\3rd Year\\2nd semester\\advanced_ai\\ai_system\\outputs\\task1_recommender\\recommendation_share_by_producer.csv',
 'producer_demand_trends.csv': 'D:\\UWE\\3rd Year\\2nd semester\\advanced_ai\\ai_system\\outputs\\task1_recommender\\producer_demand_trends.csv',
 'task1_summary.json': 'D:\\UWE\\3rd Year\\2nd semester\\advanced_ai\\ai_system\\outputs\\task1_recommender\\task1_summary.json'

In [5]:
pd.read_csv(TASK1_OUTPUT_DIR / 'recommender_metrics.csv')

,method,precision_at_3,recall_at_3,hit_rate_at_3,product_coverage,producer_diversity,cold_start_fallback_count,evaluated_customers,notes
0,global_popularity,0.116667,0.058941,0.3,0.050000,0.166667,0,60,Synthetic DESD seed data; proof-of-concept met...
1,user_frequency,0.116667,0.052362,0.3,0.933333,1.000000,0,60,Synthetic DESD seed data; proof-of-concept met...
2,frequency_recency,0.122222,0.055079,0.3,0.933333,1.000000,0,60,Synthetic DESD seed data; proof-of-concept met...


In [6]:
pd.read_csv(TASK1_OUTPUT_DIR / 'recommendation_examples.csv').head(15)

,customer_id,customer_type,method,rank,product_id,product_name,producer_id,score,reason_codes,reason_text
0,C000003,young_professional,global_popularity,1,P000132,Leek and Onion Box,PR000003,1.000000,globally_popular_product,Recommended because it is popular across the m...
1,C000003,young_professional,global_popularity,2,P000097,Aged Farmhouse Cheddar,PR000004,0.808307,globally_popular_product,Recommended because it is popular across the m...
2,C000003,young_professional,global_popularity,3,P000096,New Season Potatoes,PR000003,0.805112,globally_popular_product,Recommended because it is popular across the m...
3,C000005,community_group,global_popularity,1,P000132,Leek and Onion Box,PR000003,1.000000,globally_popular_product,Recommended because it is popular across the m...
4,C000005,community_group,global_popularity,2,P000097,Aged Farmhouse Cheddar,PR000004,0.808307,globally_popular_product,Recommended because it is popular across the m...
5,C000005,community_group,global_popularity,3,P000096,New Season Potatoes,PR000003,0.805112,globally_popular_product,Recommended because it is popular across the m...
6,C000006,restaurant,global_popularity,1,P000132,Leek and Onion Box,PR000003,1.000000,globally_popular_product,Recommended because it is popular across the m...
7,C000006,restaurant,global_popularity,2,P000097,Aged Farmhouse Cheddar,PR000004,0.808307,globally_popular_product,Recommended because it is popular across the m...
8,C000006,restaurant,global_popularity,3,P000096,New Season Potatoes,PR000003,0.805112,globally_popular_product,Recommended because it is popular across the m...
9,C000007,community_group,global_popularity,1,P000132,Leek and Onion Box,PR000003,1.000000,globally_popular_product,Recommended because it is popular across the m...


In [7]:
pd.read_csv(TASK1_OUTPUT_DIR / 'recommendation_share_by_producer.csv').head(20)

,method,producer_id,recommendation_count,recommendation_share
0,global_popularity,PR000003,120,0.666667
1,global_popularity,PR000004,60,0.333333
2,user_frequency,PR000002,15,0.083333
3,user_frequency,PR000003,20,0.111111
4,user_frequency,PR000004,20,0.111111
5,user_frequency,PR000005,22,0.122222
6,user_frequency,PR000006,8,0.044444
7,user_frequency,PR000007,9,0.050000
8,user_frequency,PR000008,15,0.083333
9,user_frequency,PR000009,17,0.094444


In [8]:
pd.read_csv(TASK1_OUTPUT_DIR / 'producer_demand_trends.csv').head(20)

,producer_id,product_id,product_name,week_start,quantity_ordered,moving_average_3w,trend_direction
0,PR000002,P000083,Somerset Discovery Apples,2026-01-12,23,23.0000,stable
1,PR000002,P000083,Somerset Discovery Apples,2026-01-19,9,16.0000,down
2,PR000002,P000083,Somerset Discovery Apples,2026-01-26,3,11.6667,down
3,PR000002,P000083,Somerset Discovery Apples,2026-02-02,18,10.0000,down
4,PR000002,P000083,Somerset Discovery Apples,2026-02-09,2,7.6667,down
5,PR000002,P000083,Somerset Discovery Apples,2026-02-16,15,11.6667,up
6,PR000002,P000083,Somerset Discovery Apples,2026-02-23,15,10.6667,down
7,PR000002,P000083,Somerset Discovery Apples,2026-03-09,7,12.3333,up
8,PR000002,P000083,Somerset Discovery Apples,2026-03-16,15,12.3333,stable
9,PR000002,P000083,Somerset Discovery Apples,2026-03-23,5,9.0000,down


## You May Also Like Discovery Evidence

Discovery is evaluated separately from quick reorder. The target is future unseen products.

In [ ]:
pd.read_csv(TASK1_OUTPUT_DIR / 'discovery_metrics.csv')

In [ ]:
pd.read_csv(TASK1_OUTPUT_DIR / 'discovery_examples.csv').head(15)